# NC coherent 1g flux weights via true-E$_\nu$ donor matching

The `NC_coherent_1g_reweighted` rows are isotropic photon-gun events reweighted to the NC coherent 1g (E$_\gamma$, cos$\theta$) shape, so they carry no neutrino and no flux universes. This notebook validates the two-step fix:

1. `coh_1g_reweighting.py` now samples a true E$_\nu$ per iso1g event from the reference coherent simulation events in the same (cos$\theta$, E$_\gamma$) bin (`coherent_1g_true_nuEnergy`).
2. `flux_weight_donor_matching.py` copies every flux column from a numu `nu_overlay` donor with matching true E$_\nu$ (random pick among the 20 nearest, fixed seed): `flux_all`/`ppfx_all` in `presel_weights_df` **and** the 13 per-knob spline-format columns (`piplus_PrimaryHadronSWCentralSplineVariation`, `horncurrent_FluxUnisim`, ...) in `spline_weights_df`. `create_rw_syst_df.py` appends the derived rows to both parquets (previously the coherent/rad-corrected rows were absent from `spline_weights_df`, so they never reached the PROfit ROOT file).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import glob, sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from src.file_locations import intermediate_files_location
from src.coh_1g_reweighting import compute_nc_coh_1g_reweighting, _load_nc_coherent_simulation, bins_energy, bins_costheta
from src.flux_weight_donor_matching import borrow_flux_weights_by_true_nu_energy, build_donor_pool
from src.systematics import create_universe_histograms
# coh_1g_reweighting.py forces the Agg backend at import, so re-select inline plotting
%matplotlib inline

presel_df_path = f"{intermediate_files_location}/presel_df_train_vars.parquet"
presel_lf = pl.scan_parquet(presel_df_path)
weight_parts = sorted(glob.glob(f"{intermediate_files_location}/presel_weights_df_*.parquet"))
print(len(weight_parts), "weight parts available:", [os.path.basename(p) for p in weight_parts])

## 1. Regenerate `coh_1g_reweighting.parquet` with the sampled true E$_\nu$

In [ ]:
compute_nc_coh_1g_reweighting(presel_lf, make_plots=False)
coh_w = pl.read_parquet(f"{intermediate_files_location}/coh_1g_reweighting.parquet")
print(coh_w.schema)
coh_w.describe()

### Sampled E$_\nu$ vs the reference coherent simulation

In [ ]:
ref_cos, ref_Eg, ref_Enu = _load_nc_coherent_simulation()

# iso1g kinematics + weights for the events that received an E_nu
iso = (presel_lf.filter(pl.col("filetype") == "isotropic_one_gamma_overlay")
       .select("run", "subrun", "event", "wc_true_leading_shower_energy", "wc_true_leading_shower_costheta", "wc_net_weight_open_data")
       .collect()
       .join(coh_w, on=["run", "subrun", "event"], how="inner"))
w_coh = (iso["wc_net_weight_open_data"] * iso["coherent_1g_weight_per_pot"]).to_numpy()
Enu_assigned = iso["coherent_1g_true_nuEnergy"].to_numpy()
print(f"{iso.height:,} iso1g events with coherent weight; assigned E_nu mean {Enu_assigned.mean():.0f} MeV, reference mean {ref_Enu.mean():.0f} MeV")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
b = np.linspace(0, 4000, 81)
ax[0].hist(ref_Enu, bins=b, density=True, histtype="step", lw=2, label="reference coherent sim (16139 evts)")
ax[0].hist(Enu_assigned, bins=b, density=True, histtype="step", lw=2, label="assigned to iso1g (unweighted)")
ax[0].hist(Enu_assigned, bins=b, weights=w_coh, density=True, histtype="step", lw=2, ls="--", label="assigned to iso1g (coherent-weighted)")
ax[0].set_xlabel("true $E_\\nu$ [MeV]"); ax[0].set_ylabel("density"); ax[0].legend()
ax[1].hist(ref_Enu, bins=b, density=True, histtype="step", lw=2, label="reference")
ax[1].hist(Enu_assigned, bins=b, weights=w_coh, density=True, histtype="step", lw=2, ls="--", label="assigned, coherent-weighted")
ax[1].set_yscale("log"); ax[1].set_xlabel("true $E_\\nu$ [MeV]"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
ax[0].hist2d(ref_Eg, ref_Enu, bins=[np.linspace(0, 1500, 60), np.linspace(0, 4000, 60)], cmin=1)
ax[0].set_title("reference coherent sim"); ax[0].set_xlabel("$E_\\gamma$ [MeV]"); ax[0].set_ylabel("true $E_\\nu$ [MeV]")
ax[1].hist2d(iso["wc_true_leading_shower_energy"].to_numpy(), Enu_assigned, weights=w_coh,
             bins=[np.linspace(0, 1500, 60), np.linspace(0, 4000, 60)], cmin=1e-30)
ax[1].set_title("iso1g with assigned $E_\\nu$ (coherent-weighted)"); ax[1].set_xlabel("$E_\\gamma$ [MeV]")
plt.tight_layout(); plt.show()

## 2. Borrow flux universes from energy-matched numu `nu_overlay` donors

In [ ]:
# recipients: the coherent rows in the current presel df, with the newly sampled E_nu.
# (Once the df is regenerated, apply_nc_coh_1g_reweighting writes this straight into wc_truth_nuEnergy.)
recipients = (presel_lf.filter(pl.col("filetype") == "NC_coherent_1g_reweighted")
              .select("filename", "run", "subrun", "event", "detailed_run_period", "wc_kine_reco_Enu", "wc_net_weight_open_data")
              .collect()
              .join(coh_w.select("run", "subrun", "event", pl.col("coherent_1g_true_nuEnergy").alias("wc_truth_nuEnergy")),
                    on=["run", "subrun", "event"], how="inner"))
print(f"{recipients.height:,} coherent recipient rows")

recipients = borrow_flux_weights_by_true_nu_energy(recipients, weight_parts, presel_df_path,
                                                   donor_filetype="nu_overlay", donor_nu_pdg=14,
                                                   weight_cols=["flux_all", "ppfx_all"], k_nearest=20, seed=42)
recipients.select("wc_truth_nuEnergy", "flux_donor_nuEnergy", "flux_donor_run", "flux_donor_event").head()

In [ ]:
off = (recipients["flux_donor_nuEnergy"] - recipients["wc_truth_nuEnergy"]).to_numpy()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(off, bins=np.linspace(-20, 20, 81)); ax[0].set_xlabel("donor $E_\\nu$ - recipient $E_\\nu$ [MeV]"); ax[0].set_yscale("log")
ax[1].scatter(recipients["wc_truth_nuEnergy"], np.abs(off), s=2, alpha=0.3)
ax[1].set_xlabel("recipient true $E_\\nu$ [MeV]"); ax[1].set_ylabel("|donor offset| [MeV]"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

## 3. How much of the flux weight is a function of E$_\nu$ alone? (donor pool)

In [ ]:
pool = build_donor_pool(weight_parts, presel_df_path, "nu_overlay", 14)
donor_full = (pl.scan_parquet(weight_parts[0]).select("filename", "run", "subrun", "event", "flux_all")
              .join(pool.lazy().select("filename", "run", "subrun", "event", "wc_truth_nuEnergy"), on=["filename", "run", "subrun", "event"], how="inner")
              .join(presel_lf.filter(pl.col("filetype") == "nu_overlay").select("filename", "run", "subrun", "event", "wc_net_weight_open_data"),
                    on=["filename", "run", "subrun", "event"], how="inner")
              .collect())
print(f"{donor_full.height:,} numu nu_overlay donors with flux_all loaded")
d_E = donor_full["wc_truth_nuEnergy"].to_numpy()
d_flux = donor_full["flux_all"].to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, a in zip([0, 1, 2], ax):
    a.scatter(d_E, [v[k] for v in d_flux], s=1, alpha=0.2)
    a.set_xlim(0, 3000); a.set_ylim(0, 2.5); a.set_xlabel("true $E_\\nu$ [MeV]"); a.set_ylabel(f"flux_all universe {k} weight")
plt.suptitle("numu nu_overlay flux weights vs true E_nu (per event the weight also depends on the parent hadron,\n"
             "so a random pick among the K nearest donors averages over that; the bin-level covariance is what must match)")
plt.tight_layout(); plt.show()

## 4. Fractional flux uncertainty: coherent recipients vs the numu nu_overlay donors

In [ ]:
def frac_unc(vals, bins, flux_arrs, weights):
    hists = create_universe_histograms(vals, bins, flux_arrs, weights, quiet=True)
    cv = np.histogram(vals, bins=bins, weights=weights)[0]
    cov = np.cov(hists, ddof=0) if hists.shape[0] > 1 else None
    std = np.sqrt(np.mean((hists - cv[:, None]) ** 2, axis=1))
    return np.divide(std, cv, out=np.zeros_like(std), where=cv > 0), cv, hists

Ebins = np.linspace(0, 3000, 101)
r_E = recipients["wc_truth_nuEnergy"].to_numpy()
r_w = recipients["wc_net_weight_open_data"].to_numpy()
r_flux = recipients["flux_all"].to_numpy()
fr_r, cv_r, h_r = frac_unc(r_E, Ebins, r_flux, r_w)
fr_d, cv_d, h_d = frac_unc(d_E, Ebins, d_flux, donor_full["wc_net_weight_open_data"].to_numpy())

fig, ax = plt.subplots(figsize=(8, 5))
c = 0.5 * (Ebins[1:] + Ebins[:-1])
ax.step(Ebins, np.r_[fr_d[0], fr_d], where="pre", lw=2, label="numu nu_overlay (donors, native flux_all)")
ax.step(Ebins, np.r_[fr_r[0], fr_r], where="pre", lw=2, ls="--", label="NC coherent 1g (borrowed flux_all)")
ax.set_xlabel("true $E_\\nu$ [MeV]"); ax.set_ylabel("fractional flux uncertainty per bin"); ax.set_ylim(0, 2); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# fractional flux uncertainty vs reco energy for the coherent rows, and the true-E_nu flux correlation matrix
Rbins = np.linspace(0, 1500, 16)
r_reco = recipients["wc_kine_reco_Enu"].to_numpy()
fr_reco, cv_reco, _ = frac_unc(r_reco, Rbins, r_flux, r_w)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].step(Rbins, np.r_[fr_reco[0], fr_reco], where="pre", lw=2)
ax[0].set_xlabel("wc_kine_reco_Enu [MeV]"); ax[0].set_ylabel("fractional flux uncertainty"); ax[0].set_ylim(0, 0.5)
ax[0].set_title("NC coherent 1g, borrowed flux universes")
diff = h_r - cv_r[:, None]
cov = diff @ diff.T / h_r.shape[1]
sd = np.sqrt(np.diag(cov)); corr = cov / np.outer(sd, sd); corr = np.nan_to_num(corr)
im = ax[1].imshow(corr, origin="lower", vmin=-1, vmax=1, cmap="RdBu_r", extent=[Ebins[0], Ebins[-1], Ebins[0], Ebins[-1]])
plt.colorbar(im, ax=ax[1]); ax[1].set_title("flux correlation matrix, coherent rows (true $E_\\nu$ bins)")
ax[1].set_xlabel("true $E_\\nu$ [MeV]"); ax[1].set_ylabel("true $E_\\nu$ [MeV]")
plt.tight_layout(); plt.show()

In [ ]:
# total normalization flux uncertainty on the coherent prediction vs on the numu donors
def total_frac(hists, cv):
    tot = hists.sum(axis=0); return np.sqrt(np.mean((tot - cv.sum()) ** 2)) / cv.sum()
print(f"total flux normalization uncertainty: coherent {total_frac(h_r, cv_r):.3f}, numu nu_overlay donors {total_frac(h_d, cv_d):.3f}")

## 5. Spline-format flux knobs (`spline_weights_df`), same donors

The per-knob flux variations used by PROfit live in the spline parquet. Borrow them with the same call (same seed → same donors as for `flux_all`) and compare the per-bin uncertainty of the 1000-universe `piplus` knob and the 3-point `horncurrent` unisim between the coherent rows and the native numu donors.

In [ ]:
spline_parts = sorted(glob.glob(f"{intermediate_files_location}/spline_weights_df_*.parquet"))
knobs = ["piplus_PrimaryHadronSWCentralSplineVariation", "kplus_PrimaryHadronFeynmanScaling", "horncurrent_FluxUnisim", "expskin_FluxUnisim"]
rec_sp = borrow_flux_weights_by_true_nu_energy(
    recipients.select("filename", "run", "subrun", "event", "wc_truth_nuEnergy", "wc_net_weight_open_data", "flux_donor_event"),
    spline_parts, presel_df_path, donor_filetype="nu_overlay", donor_nu_pdg=14, weight_cols=knobs, k_nearest=20, seed=42)
print("same donors as the flux_all pass:", (rec_sp["flux_donor_event_right"] == rec_sp["flux_donor_event"]).all()
      if "flux_donor_event_right" in rec_sp.columns else "(n/a)")

donor_sp = (pl.scan_parquet(spline_parts[0]).select("filename", "run", "subrun", "event", *knobs)
            .join(donor_full.lazy().select("filename", "run", "subrun", "event", "wc_truth_nuEnergy", "wc_net_weight_open_data"),
                  on=["filename", "run", "subrun", "event"], how="inner").collect())

fig, axes = plt.subplots(1, len(knobs), figsize=(5 * len(knobs), 4.5), sharey=True)
for k, ax in zip(knobs, axes):
    fr_dk, _, _ = frac_unc(donor_sp["wc_truth_nuEnergy"].to_numpy(), Ebins, donor_sp[k].to_numpy(), donor_sp["wc_net_weight_open_data"].to_numpy())
    fr_rk, _, _ = frac_unc(rec_sp["wc_truth_nuEnergy"].to_numpy(), Ebins, rec_sp[k].to_numpy(), rec_sp["wc_net_weight_open_data"].to_numpy())
    ax.step(Ebins, np.r_[fr_dk[0], fr_dk], where="pre", lw=2, label="numu nu_overlay (native)")
    ax.step(Ebins, np.r_[fr_rk[0], fr_rk], where="pre", lw=2, ls="--", label="NC coherent 1g (borrowed)")
    ax.set_title(k.split("_")[0]); ax.set_xlabel("true $E_\\nu$ [MeV]"); ax.set_ylim(0, 0.4)
axes[0].set_ylabel("RMS over variations / CV per bin"); axes[0].legend()
plt.tight_layout(); plt.show()